In [47]:
import os, gc, json, warnings
from typing import List, Tuple, Dict, Optional
import numpy as np
import pandas as pd
from datetime import timedelta
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore", category=UserWarning)

# =========================
# 路径配置
# =========================
MAIN_TRAIN_PATH = "train/train.csv"
BANK_TRAIN_PATH = "train/train_bank_statement.csv"
MAIN_TEST_PATH  = "testaa/testaa.csv"                 # 若不存在会跳过预测
BANK_TEST_PATH  = "testaa/testaa_bank_statement.csv"  # 若不存在会跳过预测
OUT_DIR = "output"  # 输出目录

# =========================
# 训练配置
# =========================
N_FOLDS     = 5
ITERATIONS  = 1200     
USE_GPU     = True
RANDOM_SEED = 42
TARGET_CANDIDATES = ["label","target","is_default","default","y","bad","risk_flag"]


In [48]:
# =========================
# 工具函数：主表
# =========================
def infer_target_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    # 兜底：找二分类列，名称含 default/label/target 优先
    binary_like = []
    for c in df.columns:
        if str(c).lower() in ["id","uuid","index","loan_id","user_id"]:
            continue
        vc = df[c].dropna().unique()
        if len(vc) == 2:
            binary_like.append(c)
    preferred = sorted(binary_like, key=lambda x: (0 if any(k in str(x).lower() for k in ["default","label","target"]) else 1, str(x)))
    return preferred[0] if preferred else None

def detect_datetime_cols(df: pd.DataFrame, sample_size=5000, threshold=0.8) -> List[str]:
    dt_cols = []
    for c in df.columns:
        s = df[c]
        if s.dtype == "object" or np.issubdtype(s.dtype, np.integer) or np.issubdtype(s.dtype, np.floating):
            sample = s.dropna().astype(str).head(sample_size)
            if sample.empty: 
                continue
            parsed = pd.to_datetime(sample, errors="coerce", infer_datetime_format=True)
            if parsed.notna().mean() >= threshold:
                dt_cols.append(c)
    return dt_cols

def add_time_features_for_main(df: pd.DataFrame, dt_cols: List[str]) -> Tuple[pd.DataFrame, List[str]]:
    """在主表上对指定时间列衍生时间特征（并删除原时间列）。"""
    added = []
    for c in dt_cols:
        s = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
        df[f"{c}_year"] = s.dt.year
        df[f"{c}_month"] = s.dt.month
        df[f"{c}_day"] = s.dt.day
        df[f"{c}_dow"] = s.dt.dayofweek
        df[f"{c}_is_month_start"] = s.dt.is_month_start.astype("Int8")
        df[f"{c}_is_month_end"] = s.dt.is_month_end.astype("Int8")
        df[f"{c}_quarter"] = s.dt.quarter
        df[f"{c}_hour"] = s.dt.hour
        if s.notna().any():
            min_ts = s.min()
            df[f"{c}_days_since_min"] = (s - min_ts).dt.days
        added += [f"{c}_year", f"{c}_month", f"{c}_day", f"{c}_dow",
                  f"{c}_is_month_start", f"{c}_is_month_end",
                  f"{c}_quarter", f"{c}_hour", f"{c}_days_since_min"]
        df.drop(columns=[c], inplace=True)
    return df, added

def add_missing_indicators(df: pd.DataFrame, thr_low=0.03, thr_high=0.95) -> List[str]:
    added = []
    miss_rate = df.isna().mean()
    for c, r in miss_rate.items():
        if thr_low <= r <= thr_high:
            ind = f"{c}_isna"
            df[ind] = df[c].isna().astype("Int8")
            added.append(ind)
    return added

def pick_first_col(cols: List[str], patterns: List[str]) -> Optional[str]:
    for p in patterns:
        for c in cols:
            if p in c.lower():
                return c
    return None

def safe_ratio(u: pd.Series, v: pd.Series) -> pd.Series:
    out = u.astype(float) / np.where(v.astype(float) == 0, np.nan, v.astype(float))
    return out.clip(lower=-1e6, upper=1e6)

def build_ratio_features_for_main(df: pd.DataFrame) -> List[str]:
    """主表：语义驱动的有限比率特征（避免噪声过多）。"""
    added = []
    cols = df.columns.tolist()

    income_col   = pick_first_col(cols, ["income","salary"])
    debt_col     = pick_first_col(cols, ["debt","liab"])
    balance_col  = pick_first_col(cols, ["balance","bal"])
    limit_col    = pick_first_col(cols, ["limit","credit_limit"])
    amount_col   = pick_first_col(cols, ["amount","amt","principal","loan_am","funded"])
    payment_col  = pick_first_col(cols, ["payment","installment","repay"])
    bill_col     = pick_first_col(cols, ["bill"])

    # 避免与已存在的 balance_limit 重复
    has_balance_limit = pick_first_col(cols, ["balance_limit"]) is not None

    if debt_col and income_col:
        name = "debt_to_income"
        df[name] = safe_ratio(df[debt_col], df[income_col]); added.append(name)
    if balance_col and limit_col and not has_balance_limit:
        name = "balance_to_limit"
        df[name] = safe_ratio(df[balance_col], df[limit_col]); added.append(name)
    if amount_col and income_col:
        name = "amount_to_income"
        df[name] = safe_ratio(df[amount_col], df[income_col]); added.append(name)
    if payment_col and income_col:
        name = "payment_to_income"
        df[name] = safe_ratio(df[payment_col], df[income_col]); added.append(name)
    if payment_col and balance_col:
        name = "payment_to_balance"
        df[name] = safe_ratio(df[payment_col], df[balance_col]); added.append(name)
    if bill_col and income_col:
        name = "bill_to_income"
        df[name] = safe_ratio(df[bill_col], df[income_col]); added.append(name)

    return added

def cast_special_categoricals(df: pd.DataFrame) -> List[str]:
    """把邮编等数值形态的列转为字符串类别。"""
    cat_special = []
    for c in df.columns:
        name = c.lower()
        if any(k in name for k in ["zip","zipcode","postal"]):
            df[c] = df[c].astype("Int64").astype(str)
            cat_special.append(c)
    return cat_special

def get_low_card_int_as_cat(df: pd.DataFrame, target: Optional[str]) -> List[str]:
    res = []
    for c in df.columns:
        if c == target: 
            continue
        if np.issubdtype(df[c].dtype, np.integer):
            uniq = df[c].nunique(dropna=True)
            if 0 < uniq <= max(100, int(0.03 * len(df))):
                res.append(c)
    return res

# =========================
# 工具函数：银行流水
# =========================
def detect_epoch_unit(series: pd.Series) -> str:
    s = series.dropna().astype(np.int64)
    if s.empty: return "s"
    m = int(s.head(2000).median())
    return "ms" if m > 10**12 else "s"

def parse_time_column(s: pd.Series) -> pd.Series:
    if np.issubdtype(s.dtype, np.number):
        unit = detect_epoch_unit(s)
        return pd.to_datetime(s, unit=unit, errors="coerce")
    return pd.to_datetime(s.astype(str), errors="coerce", infer_datetime_format=True)

def agg_block_for_bank(df: pd.DataFrame, prefix: str, id_col: str, global_max_time) -> pd.DataFrame:
    # 先拷贝并准备列，再 groupby，避免 KeyError
    df = df.copy()

    # 交易计数（先占位，后面 join 更多指标）
    # 注：先不分组，等金额/时间列准备好后再分组
    # 这里不急着构造 feat，直接等 groupby 后统一生成

    # 如果有 _signed_amount，就先准备绝对值列
    has_signed = ("_signed_amount" in df.columns)
    if has_signed:
        df["_abs_amount"] = df["_signed_amount"].abs()

    # 时间列相关准备
    has_time = ("_time" in df.columns) and df["_time"].notna().any()

    # *** 现在再 groupby，确保能看到刚新增的列 ***
    g = df.groupby(id_col, observed=True)

    # 先做交易笔数
    feat = pd.DataFrame({f"{prefix}txn_cnt": g.size()})

    # 金额统计
    if has_signed:
        g_amt = g["_signed_amount"]
        g_abs = g["_abs_amount"]

        feat[f"{prefix}net_sum"]  = g_amt.sum()
        feat[f"{prefix}net_mean"] = g_amt.mean()
        feat[f"{prefix}net_std"]  = g_amt.std()
        feat[f"{prefix}abs_mean"] = g_abs.mean()
        feat[f"{prefix}abs_p50"]  = g_abs.median()
        feat[f"{prefix}abs_p90"]  = g_abs.quantile(0.9)
        feat[f"{prefix}abs_max"]  = g_abs.max()
        feat[f"{prefix}abs_min"]  = g_abs.min()

        # 流入 / 流出
        gin  = df.loc[df["_signed_amount"] > 0].groupby(id_col)["_signed_amount"]
        gout = df.loc[df["_signed_amount"] < 0].groupby(id_col)["_signed_amount"]
        feat[f"{prefix}in_cnt"]   = gin.size()
        feat[f"{prefix}in_sum"]   = gin.sum().abs()
        feat[f"{prefix}in_mean"]  = gin.mean().abs()
        feat[f"{prefix}out_cnt"]  = gout.size()
        feat[f"{prefix}out_sum"]  = gout.sum().abs()
        feat[f"{prefix}out_mean"] = gout.mean().abs()

        # 比率与大额占比（阈值用该子集的 P95）
        feat[f"{prefix}out_in_ratio"] = feat[f"{prefix}out_sum"] / (feat[f"{prefix}in_sum"] + 1e-6)
        feat[f"{prefix}avg_ticket"]   = feat[f"{prefix}abs_mean"]

        thr = df["_abs_amount"].quantile(0.95)
        large = df[df["_abs_amount"] >= thr]
        feat[f"{prefix}large_cnt_ratio"] = (
            large.groupby(id_col).size() / (feat[f"{prefix}txn_cnt"] + 1e-6)
        ).reindex(feat.index).fillna(0)

    # 时间行为统计
    if has_time:
        df2 = df[df["_time"].notna()].copy()
        df2["_dow"]  = df2["_time"].dt.dayofweek
        df2["_hour"] = df2["_time"].dt.hour
        g2 = df2.groupby(id_col)

        feat[f"{prefix}wknd_ratio"]  = g2["_dow"].apply(lambda s: (s >= 5).mean())
        feat[f"{prefix}night_ratio"] = g2["_hour"].apply(lambda s: ((s >= 0) & (s < 6)).mean())

        df2["_date"] = df2["_time"].dt.date
        feat[f"{prefix}active_days"]   = g2["_date"].nunique()

        def avg_gap(s):
            s = pd.to_datetime(s.sort_values().astype("datetime64[ns]"), errors="coerce")
            if s.size <= 1: 
                return np.nan
            gaps = (s.values[1:] - s.values[:-1]) / np.timedelta64(1, 'D')
            return np.mean(gaps)

        feat[f"{prefix}avg_gap_days"]  = g2["_time"].apply(avg_gap)

        if global_max_time is not None:
            last_t = g2["_time"].max()
            feat[f"{prefix}days_since_last"] = (global_max_time - last_t).dt.days

    return feat

def build_bank_features(bs: pd.DataFrame,
                        id_col: str = "id",
                        time_col: str = "time",
                        dir_col: str  = "direction",
                        amt_col: str  = "amount",
                        windows: List[int] = [30,60,90]) -> pd.DataFrame:
    """银行流水聚合为客户级特征。"""
    assert id_col in bs.columns, "银行流水缺少id列"
    # 时间解析
    if time_col in bs.columns:
        bs["_time"] = parse_time_column(bs[time_col])
    else:
        bs["_time"] = pd.NaT
    # 有符号金额
    if amt_col in bs.columns:
        if dir_col in bs.columns:
            bs["_signed_amount"] = np.where(bs[dir_col].astype(int)==0,
                                            bs[amt_col].astype(float),
                                            -bs[amt_col].astype(float))
        else:
            bs["_signed_amount"] = bs[amt_col].astype(float)
    else:
        bs["_signed_amount"] = np.nan

    global_max_time = bs["_time"].max() if bs["_time"].notna().any() else None

    # 全量
    aggs = [agg_block_for_bank(bs, "all_", id_col, global_max_time)]
    # 窗口
    if global_max_time is not None and windows:
        for w in windows:
            bs_w = bs[bs["_time"] >= global_max_time - timedelta(days=w)].copy()
            aggs.append(agg_block_for_bank(bs_w, f"win{w}d_", id_col, global_max_time))

    bank_feat = aggs[0]
    for a in aggs[1:]:
        bank_feat = bank_feat.join(a, how="outer")
    bank_feat = bank_feat.reset_index().rename(columns={id_col: "id"})

    # 计数/和类 → 0；其余保留 NaN（CatBoost可处理）
    for c in bank_feat.columns:
        cl = c.lower()
        if any(k in cl for k in ["_cnt","_sum","active_days"]):
            bank_feat[c] = bank_feat[c].fillna(0)
    return bank_feat

# =========================
# 评估函数
# =========================
def ks_score(y_true, y_prob) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(np.abs(tpr - fpr)))


In [63]:

# =========================
# 主流程
# =========================
from sklearn.model_selection import train_test_split
# 读取训练数据
assert os.path.exists(MAIN_TRAIN_PATH), f"未找到主训练集：{MAIN_TRAIN_PATH}"
assert os.path.exists(BANK_TRAIN_PATH), f"未找到银行流水训练集：{BANK_TRAIN_PATH}"
main_train = pd.read_csv(MAIN_TRAIN_PATH, low_memory=False)
bank_train = pd.read_csv(BANK_TRAIN_PATH, low_memory=False)

# 目标列
target = infer_target_col(main_train, TARGET_CANDIDATES)
assert target is not None, "未能自动识别目标列，请在 TARGET_CANDIDATES 中加入正确列名"
print(f"[INFO] Target: {target}")

# 主表：时间列探测与衍生
dt_cols_main = detect_datetime_cols(main_train)
if dt_cols_main:
    print(f"[INFO] Main datetime cols: {dt_cols_main}")
    main_train, added_time = add_time_features_for_main(main_train, dt_cols_main)
    print(f"[INFO] Added time features: {len(added_time)}")
else:
    print("[INFO] Main: no datetime columns detected.")

# 主表：比率特征
added_ratio = build_ratio_features_for_main(main_train)
print(f"[INFO] Main ratio features: {added_ratio}")

# 主表：类别列识别
obj_cols = [c for c in main_train.columns if main_train[c].dtype == "object" and c != target]
low_card_int = get_low_card_int_as_cat(main_train, target)
special_cat = cast_special_categoricals(main_train)
obj_cols = sorted(list(set(obj_cols + special_cat)))
cat_cols_train = sorted(list(set(obj_cols + low_card_int)))

# 类别缺失填充
for c in cat_cols_train:
    if c in main_train.columns:
        main_train[c] = main_train[c].astype("string").fillna("Unknown")

# 主表：缺失指示变量（根据训练集统计）
miss_inds = add_missing_indicators(main_train)
print(f"[INFO] Main missing indicators: {len(miss_inds)}")

# 银行流水 → 客户聚合特征
bank_feat_train = build_bank_features(bank_train, id_col="id", time_col="time", dir_col="direction", amt_col="amount", windows=[30,60,90])
bank_feat_path = os.path.join(OUT_DIR, "bank_features_train.csv")
bank_feat_train.to_csv(bank_feat_path, index=False)

# 合并训练集
train = main_train.merge(bank_feat_train, on="id", how="left")

# 再对合并后的“新列”添加缺失指示（与前面一致阈值）
add_missing_indicators(train)  # 可能新增少量缺失指示

# 组装训练特征矩阵
features = [c for c in train.columns if c != target]
X = train[features].copy()
y = train[target].astype(int).values

# CatBoost 的类别列索引（以训练集为准）
cat_idx = [X.columns.get_loc(c) for c in cat_cols_train if c in X.columns]

# 不平衡权重
pos_rate = float(np.mean(y))
w_pos = 0.5 / max(pos_rate, 1e-6)
w_neg = 0.5 / max(1.0 - pos_rate, 1e-6)
class_weights = [w_neg, w_pos]
print(f"[INFO] Class weights: {class_weights} (pos_rate={pos_rate:.6f})")

# 训练参数
params = dict(
    cat_features=cat_idx,
    loss_function="Logloss",
    eval_metric="AUC",
    depth=7,
    learning_rate=0.04,
    l2_leaf_reg=5.0,
    iterations=ITERATIONS,
    random_seed=RANDOM_SEED,
    #od_type="Iter",
    #od_wait=max(50, ITERATIONS//12),
    #od_wait=200,
    class_weights=class_weights,
    verbose=200,
    allow_const_label=True,
    task_type="GPU" if USE_GPU else "CPU",
)

# 交叉验证
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
oof = np.zeros(len(X))
fold_metrics = []
'''
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
    va_pool = Pool(X_va, y_va, cat_features=cat_idx)

    clf = CatBoostClassifier(**params)
    clf.fit(tr_pool, eval_set=va_pool, use_best_model=True)
    pred = clf.predict_proba(va_pool)[:,1]
    oof[va_idx] = pred

    auc = roc_auc_score(y_va, pred)
    ks  = ks_score(y_va, pred)
    yhat= (pred >= 0.5).astype(int)
    f1  = f1_score(y_va, yhat)
    p   = precision_score(y_va, yhat)
    r   = recall_score(y_va, yhat)
    fold_metrics.append({"fold": fold, "AUC": auc, "KS": ks, "F1@0.5": f1, "Prec@0.5": p, "Rec@0.5": r})
    # 每折重要性（可选）
    fi = pd.DataFrame({"feature": X.columns, "importance": clf.get_feature_importance(tr_pool)})
    fi.sort_values("importance", ascending=False).to_csv(os.path.join(OUT_DIR, f"feature_importance_fold{fold}.csv"), index=False)

    del tr_pool, va_pool, clf
    gc.collect()

metrics_df = pd.DataFrame(fold_metrics)
metrics_df.to_csv(os.path.join(OUT_DIR, "cv_metrics.csv"), index=False)
pd.DataFrame({"oof_pred": oof, "label": y}).to_csv(os.path.join(OUT_DIR, "oof_preds.csv"), index=False)
# 全量训练最终模型
final_clf = CatBoostClassifier(**params)
full_pool = Pool(X, y, cat_features=cat_idx)
final_clf.fit(full_pool,use_best_model=True)
final_clf.save_model(os.path.join(OUT_DIR, "catboost_model_with_bank.cbm"))
'''
final_clf = CatBoostClassifier(**params)
X_train, X_validation, y_train, y_validation = train_test_split(X, y, test_size=0.2 , random_state=2000)
final_clf.fit(X_train, y_train,eval_set = (X_validation,y_validation))
fi_all = pd.DataFrame({"feature": X.columns, "importance": final_clf.get_feature_importance(full_pool)})
fi_all.sort_values("importance", ascending=False).to_csv(os.path.join(OUT_DIR, "feature_importance.csv"), index=False)
# 保存特征规范（用于测试集对齐）    
spec = {
    "features": X.columns.tolist(),
    "cat_features": [X.columns[i] for i in cat_idx],
    "dt_cols_main": dt_cols_main,          # 训练阶段识别到的主表时间列名（供测试复用）
    "miss_indicator_cols": [c for c in train.columns if c.endswith("_isna")],
    "ratio_cols_main": added_ratio,
}
with open(os.path.join(OUT_DIR, "features_used.json"), "w", encoding="utf-8") as f:
    json.dump(spec, f, ensure_ascii=False, indent=2)



[INFO] Target: label
[INFO] Main: no datetime columns detected.
[INFO] Main ratio features: ['payment_to_balance']
[INFO] Main missing indicators: 1
[INFO] Class weights: [0.6134153055606533, 2.7042880258899675] (pos_rate=0.184892)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6406832	best: 0.6406832 (0)	total: 69.2ms	remaining: 1m 22s
200:	test: 0.6735126	best: 0.6735126 (200)	total: 11.5s	remaining: 57.2s
400:	test: 0.6746147	best: 0.6746327 (395)	total: 23.2s	remaining: 46.3s
600:	test: 0.6745306	best: 0.6747465 (445)	total: 37.6s	remaining: 37.5s
800:	test: 0.6750087	best: 0.6751940 (760)	total: 49.2s	remaining: 24.5s
1000:	test: 0.6739000	best: 0.6751940 (760)	total: 1m	remaining: 12.1s
1199:	test: 0.6733873	best: 0.6751940 (760)	total: 1m 12s	remaining: 0us
bestTest = 0.6751939654
bestIteration = 760
Shrink model to first 761 iterations.


In [64]:
# =========================
# 测试集处理与预测
# =========================
if os.path.exists(MAIN_TEST_PATH) and os.path.exists(BANK_TEST_PATH):
    print("[INFO] Found test files. Start test processing...")
    test_main = pd.read_csv(MAIN_TEST_PATH, low_memory=False)
    test_bank = pd.read_csv(BANK_TEST_PATH, low_memory=False)

    # 主表：按训练规范进行相同的处理
    # 1) 时间特征（使用训练时检测到的 dt_cols_main）
    if spec["dt_cols_main"]:
        for c in list(spec["dt_cols_main"]):
            if c in test_main.columns:
                s = pd.to_datetime(test_main[c], errors="coerce", infer_datetime_format=True)
                test_main[f"{c}_year"] = s.dt.year
                test_main[f"{c}_month"] = s.dt.month
                test_main[f"{c}_day"] = s.dt.day
                test_main[f"{c}_dow"] = s.dt.dayofweek
                test_main[f"{c}_is_month_start"] = s.dt.is_month_start.astype("Int8")
                test_main[f"{c}_is_month_end"] = s.dt.is_month_end.astype("Int8")
                test_main[f"{c}_quarter"] = s.dt.quarter
                test_main[f"{c}_hour"] = s.dt.hour
                if s.notna().any():
                    min_ts = s.min()
                    test_main[f"{c}_days_since_min"] = (s - min_ts).dt.days
                test_main.drop(columns=[c], inplace=True, errors="ignore")

    # 2) 同样的语义比率（若训练构造过）
    #    直接再次调用构造函数也可以；为了与训练一致，这里无条件再构造一遍（新列不冲突）。
    build_ratio_features_for_main(test_main)

    # 3) 特殊类别列 & 低基数整数处理（类型转换后置）
    #    以训练时识别的 cat_features 为准：这些列在 test 中统一转字符串并填充 Unknown
    for c in spec["cat_features"]:
        if c in test_main.columns:
            test_main[c] = test_main[c].astype("string").fillna("Unknown")
        else:
            # 测试中缺失的类别列也要补列，填为 Unknown
            test_main[c] = "Unknown"

    # 4) 缺失指示变量（与训练相同的列名集合）
    for ind_col in spec["miss_indicator_cols"]:
        base_col = ind_col[:-5]  # 去掉 _isna
        if base_col in test_main.columns:
            test_main[ind_col] = test_main[base_col].isna().astype("Int8")
        else:
            # 若基础列不存在，则指示置0
            test_main[ind_col] = 0

    # 银行流水 → 聚合
    bank_feat_test = build_bank_features(test_bank, id_col="id", time_col="time", dir_col="direction", amt_col="amount", windows=[30,60,90])
    bank_feat_test.to_csv(os.path.join(OUT_DIR, "bank_features_test.csv"), index=False)

    # 合并
    test_merged = test_main.merge(bank_feat_test, on="id", how="left")

    # 计数/和类列（新合并的）缺失补0（防止空列影响）
    for c in test_merged.columns:
        cl = c.lower()
        if any(k in cl for k in ["_cnt","_sum","active_days"]):
            test_merged[c] = test_merged[c].fillna(0)

    # 与训练特征对齐（缺列补、顺序一致）
    used_features = spec["features"]
    for c in used_features:
        if c not in test_merged.columns:
            # 类别列缺失 → Unknown；数值列缺失 → NaN（CatBoost可处理）
            if c in spec["cat_features"]:
                test_merged[c] = "Unknown"
            else:
                test_merged[c] = np.nan
    X_test = test_merged[used_features].copy()

    # 构建 Pool 并预测
    cat_idx_test = [X_test.columns.get_loc(c) for c in spec["cat_features"] if c in X_test.columns]
    test_pool = Pool(X_test, cat_features=cat_idx_test)
    test_pred = final_clf.predict_proba(test_pool)[:, 1]

    # 输出预测（id + label）
    if "id" in test_merged.columns:
        sub = pd.DataFrame({"id": test_merged["id"], "label": test_pred})
    else:
        sub = pd.DataFrame({"label": test_pred})
    sub_path = os.path.join(OUT_DIR, "test_pred.csv")
    sub.to_csv(sub_path, index=False)
    print(f"[DONE] Test predictions saved to: {sub_path}")
else:
    print("[WARN] 未检测到测试集文件（/mnt/data/test.csv & /mnt/data/test_bank_statement.csv），已跳过预测阶段。")

print("\n[OUTPUT]")
print(f" - {os.path.join(OUT_DIR,'bank_features_train.csv')}（银行流水聚合特征-训练）")
print(f" - {os.path.join(OUT_DIR,'cv_metrics.csv')}（CV评估）")
print(f" - {os.path.join(OUT_DIR,'oof_preds.csv')}（OOF预测）")
print(f" - {os.path.join(OUT_DIR,'feature_importance.csv')} & feature_importance_fold*.csv（特征重要性）")
print(f" - {os.path.join(OUT_DIR,'catboost_model_with_bank.cbm')}（最终模型）")
print(f" - {os.path.join(OUT_DIR,'features_used.json')}（特征规范，用于测试对齐）")
print(f" - {os.path.join(OUT_DIR,'bank_features_test.csv')}（若存在测试流水）")
print(f" - {os.path.join(OUT_DIR,'test_pred.csv')}（若存在测试集）")

[INFO] Found test files. Start test processing...
[DONE] Test predictions saved to: output\test_pred.csv

[OUTPUT]
 - output\bank_features_train.csv（银行流水聚合特征-训练）
 - output\cv_metrics.csv（CV评估）
 - output\oof_preds.csv（OOF预测）
 - output\feature_importance.csv & feature_importance_fold*.csv（特征重要性）
 - output\catboost_model_with_bank.cbm（最终模型）
 - output\features_used.json（特征规范，用于测试对齐）
 - output\bank_features_test.csv（若存在测试流水）
 - output\test_pred.csv（若存在测试集）
